In [ ]:
# # Cease & Desist Document Processing System (LangChain + LangGraph)
# 
# This notebook demonstrates a multi-agent workflow for automating the classification and processing of Cease & Desist documents, as described in the capstone project. It is Colab-ready and uses the same libraries, API key setup, and agent patterns as previous labs.
# 
# **Features:**
# - Document classification (Cease, Uncertain, Irrelevant)
# - Multi-agent orchestration (classification, database, archiving, audit, HITL)
# - SQLite for structured storage
# - CSV for audit logging
# - Human-in-the-loop (HITL) for uncertain cases
# - Uses sample PDFs from the provided folders
# 
# ---
# 
# **Instructions:**
# - Run each cell in order.
# - Set your GROQ API key when prompted (same as previous labs).
# - All dependencies are installed automatically.
# - You can upload your own PDFs or use the provided samples.
# 
# ---
# 
# *Created for the Day 5 Capstone Project (LangChain/LangGraph track)*

In [ ]:
# --- 1. Install dependencies (Colab)
!pip install -qU langchain-groq langgraph langchain-community pysqlite3-binary pdfplumber

# --- 2. Set up environment variables (GROQ API key)
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
print("Environment ready. If you see no errors above, continue.")

In [ ]:
# Colab file upload widget
from google.colab import files
import glob

uploaded = files.upload()  # User selects files

# List all PDFs in the current directory after upload
pdf_files = glob.glob('*.pdf')
print(f"Found {len(pdf_files)} PDF(s):", pdf_files)

# Use uploaded files for demo
DEMO_FILES = pdf_files[:3]  # Use first 3 for demo

## 2A. Upload Your PDF Files (Colab)
If running in Colab, use the cell below to upload your sample PDFs. You can select multiple files at once.

## 3. Load Sample Documents

This notebook uses sample PDFs from the `Sample Docs` and `data/pdfs` folders. You can also upload your own PDFs using the Colab file upload widget if desired.

For demonstration, we'll process a few sample files automatically.

In [ ]:
# List sample PDF files from upload or fallback to local folders
try:
    DEMO_FILES
except NameError:
    import glob
    sample_paths = glob.glob('Sample Docs/*.pdf') + glob.glob('data/pdfs/*.pdf')
    print(f"Found {len(sample_paths)} sample PDFs.")
    for i, path in enumerate(sample_paths[:5]):
        print(f"{i+1}. {path}")
    DEMO_FILES = sample_paths[:3]  # fallback if not running in Colab
else:
    print(f"Using uploaded files: {DEMO_FILES}")

## 4. PDF Text Extraction Utility

We'll use `pdfplumber` to extract text from the sample PDFs. This utility will be used by the document loader agent.

In [ ]:
import pdfplumber

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

# Test extraction on a sample file
for file in DEMO_FILES:
    print(f"\n--- {file} ---\n")
    print(extract_text_from_pdf(file)[:500])  # Print first 500 chars

## 5. Agent System Setup

We'll now define the agents and workflow using LangChain and LangGraph. The system will include:
- Document Loader Agent
- Classification Agent
- Database Agent (SQLite)
- Archiving Agent (CSV)
- Audit Agent (CSV)
- HITL Agent (manual input for demo)

We'll use the same LLM setup as previous labs (GROQ API via LangChain).

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_community.utilities import SQLDatabase
import sqlite3
import csv
import datetime

# --- LLM Setup (GROQ, same as previous labs) ---
llm = ChatGroq(temperature=0.2)

# --- SQLite DB Setup ---
DB_PATH = "cease_requests.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute('''CREATE TABLE IF NOT EXISTS cease_requests (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    received_date TEXT,
    document_name TEXT,
    extracted_details TEXT
)''')
conn.commit()

# --- CSV Archive & Audit Setup ---
ARCHIVE_CSV = "irrelevant_docs.csv"
AUDIT_CSV = "audit_log.csv"

# Write headers if files don't exist
for fname, headers in [
    (ARCHIVE_CSV, ["received_date", "document_name"]),
    (AUDIT_CSV, ["timestamp", "document_name", "action", "explanation"])
]:
    try:
        with open(fname, 'x', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(headers)
    except FileExistsError:
        pass

In [ ]:
# --- Agent Definitions ---

# 1. Document Loader Agent
class DocumentLoaderAgent:
    def __init__(self):
        pass
    def load(self, file_path):
        text = extract_text_from_pdf(file_path)
        return text

# 2. Classification Agent (LLM)
class ClassificationAgent:
    def __init__(self, llm):
        self.llm = llm
        self.prompt = ChatPromptTemplate.from_messages([
            SystemMessage(content="""
You are a compliance assistant. Classify the following document as one of:
- Cease: A valid cease & desist request
- Uncertain: Not sure, needs human review
- Irrelevant: Not a cease request

Respond with only one word: Cease, Uncertain, or Irrelevant.
"""),
            HumanMessage(content="{document_text}")
        ])
    def classify(self, document_text):
        chain = self.prompt | self.llm | StrOutputParser()
        result = chain.invoke({"document_text": document_text})
        return result.strip().capitalize()

# 3. Database Agent
class DatabaseAgent:
    def __init__(self, db_path):
        self.db_path = db_path
    def store(self, received_date, document_name, extracted_details):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("INSERT INTO cease_requests (received_date, document_name, extracted_details) VALUES (?, ?, ?)",
                       (received_date, document_name, extracted_details))
        conn.commit()
        conn.close()

# 4. Archiving Agent
class ArchivingAgent:
    def __init__(self, csv_path):
        self.csv_path = csv_path
    def archive(self, received_date, document_name):
        with open(self.csv_path, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([received_date, document_name])

# 5. Audit Agent
class AuditAgent:
    def __init__(self, csv_path):
        self.csv_path = csv_path
    def log(self, document_name, action, explanation):
        with open(self.csv_path, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                datetime.datetime.now().isoformat(),
                document_name,
                action,
                explanation
            ])

# 6. HITL Agent (manual input for demo)
class HITLAgent:
    def review(self, document_name, document_text):
        print(f"\n--- HUMAN REVIEW REQUIRED for {document_name} ---\n")
        print(document_text[:1000])
        decision = input("Classify as Cease or Irrelevant? (type 'Cease' or 'Irrelevant'): ")
        return decision.strip().capitalize()

## 6. Main Workflow: Orchestrate Agents

This cell runs the full workflow for each sample document:
- Loads the document
- Classifies it
- Processes according to class (DB, archive, or HITL)
- Logs all actions

You can modify `DEMO_FILES` to process more or different files.

In [ ]:
# Instantiate agents
loader = DocumentLoaderAgent()
classifier = ClassificationAgent(llm)
db_agent = DatabaseAgent(DB_PATH)
archive_agent = ArchivingAgent(ARCHIVE_CSV)
audit_agent = AuditAgent(AUDIT_CSV)
hitl_agent = HITLAgent()

for file_path in DEMO_FILES:
    doc_name = file_path.split('/')[-1]
    received_date = datetime.date.today().isoformat()
    text = loader.load(file_path)
    audit_agent.log(doc_name, "Loaded", "Document loaded and text extracted.")
    
    classification = classifier.classify(text)
    audit_agent.log(doc_name, "Classified", f"Classified as {classification}.")
    print(f"\n{doc_name}: Classified as {classification}")
    
    if classification == "Cease":
        db_agent.store(received_date, doc_name, text[:500])  # Store first 500 chars for demo
        audit_agent.log(doc_name, "Stored in DB", "Cease request details saved.")
        print(f"Stored in database.")
    elif classification == "Irrelevant":
        archive_agent.archive(received_date, doc_name)
        audit_agent.log(doc_name, "Archived", "Irrelevant document archived.")
        print(f"Archived to CSV.")
    elif classification == "Uncertain":
        decision = hitl_agent.review(doc_name, text)
        audit_agent.log(doc_name, "HITL Review", f"Human classified as {decision}.")
        if decision == "Cease":
            db_agent.store(received_date, doc_name, text[:500])
            audit_agent.log(doc_name, "Stored in DB", "Cease request details saved after HITL.")
            print(f"Stored in database after HITL.")
        else:
            archive_agent.archive(received_date, doc_name)
            audit_agent.log(doc_name, "Archived", "Archived after HITL.")
            print(f"Archived to CSV after HITL.")
    else:
        audit_agent.log(doc_name, "Error", f"Unknown classification: {classification}")
        print(f"Unknown classification: {classification}")

## 7. View Results

You can inspect the SQLite database and CSV files generated by the workflow. The following cells show how to read and display their contents.

In [ ]:
import pandas as pd

# View database entries
def show_db():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT * FROM cease_requests", conn)
    conn.close()
    return df

# View archive CSV
def show_archive():
    return pd.read_csv(ARCHIVE_CSV)

# View audit log
def show_audit():
    return pd.read_csv(AUDIT_CSV)

print("Database entries:")
display(show_db())
print("\nArchived documents:")
display(show_archive())
print("\nAudit log:")
display(show_audit())

---

## 8. Notes & Next Steps
- You can upload additional PDFs and rerun the workflow cell.
- For a real deployment, replace the HITL agent's `input()` with a UI or workflow tool.
- The LLM prompt for classification can be improved with examples/few-shot learning for higher accuracy.
- All files (DB, CSVs) are saved in the Colab working directory and can be downloaded.

---

**End of Capstone Demo Notebook**

## 9. (Bonus) Integrating RAG (Retrieval-Augmented Generation)

We'll enhance the workflow by adding a RAG step: before classification, the agent will retrieve similar past documents from a vector store and use their content as additional context for the LLM. This can improve classification accuracy, especially for edge cases.

In [ ]:
# --- RAG Setup: Vector Store for Document Retrieval ---
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# Build or load a vector store from all processed documents (for demo, use extracted text from DEMO_FILES)
vector_store_path = "rag_vector_store.faiss"
embedding = OpenAIEmbeddings()

# Prepare documents for vector store (use extracted text and file names)
docs = []
metadatas = []
for file_path in DEMO_FILES:
    text = extract_text_from_pdf(file_path)
    docs.append(text)
    metadatas.append({"source": file_path})

# Create FAISS vector store
vectorstore = FAISS.from_texts(docs, embedding=embedding, metadatas=metadatas)
vectorstore.save_local(vector_store_path)

# Function to retrieve similar docs
def retrieve_similar_docs(query, k=2):
    vs = FAISS.load_local(vector_store_path, embedding, allow_dangerous_deserialization=True)
    results = vs.similarity_search(query, k=k)
    return results

In [ ]:
# --- RAG-Enhanced Classification Agent ---
class RAGClassificationAgent(ClassificationAgent):
    def __init__(self, llm, retriever):
        super().__init__(llm)
        self.retriever = retriever
        self.prompt = ChatPromptTemplate.from_messages([
            SystemMessage(content="""
You are a compliance assistant. Classify the following document as one of:
- Cease: A valid cease & desist request
- Uncertain: Not sure, needs human review
- Irrelevant: Not a cease request

You are also provided with similar past documents for reference. Use them to help your decision.
Respond with only one word: Cease, Uncertain, or Irrelevant.
"""),
            HumanMessage(content="Document to classify:\n{document_text}\n\nSimilar past documents:\n{retrieved_context}")
        ])
    def classify(self, document_text):
        # Retrieve similar docs
        similar = self.retriever(document_text, k=2)
        retrieved_context = "\n---\n".join([doc.page_content for doc in similar])
        chain = self.prompt | self.llm | StrOutputParser()
        result = chain.invoke({
            "document_text": document_text,
            "retrieved_context": retrieved_context
        })
        return result.strip().capitalize()

In [ ]:
# --- Demo: Use RAG-Enhanced Agent on a Sample Document ---
rag_classifier = RAGClassificationAgent(llm, retrieve_similar_docs)

for file_path in DEMO_FILES:
    doc_name = file_path.split('/')[-1]
    text = extract_text_from_pdf(file_path)
    print(f"\nRAG Classification for {doc_name}:")
    result = rag_classifier.classify(text)
    print(f"Predicted class: {result}")